<a href="https://colab.research.google.com/github/giolessa/100-Days-of-Data/blob/main/day_05_new_columns_np_where_map_apply.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EXPLANATION

  1. Creating a new column is just assigning to a key that doesn't exist yet: df["new"] = .... When the calculation can be done with direct operations between columns (df["a"] * df["b"]), that's the best approach because it's fast and simple.

  2. np.where(condition, value_if_true, value_if_false) is an if/else applied to the entire column at once. It's the equivalent of Excel's IF and a two-branch CASE WHEN in SQL. Requires import numpy as np.

  2. np.select(list_of_conditions, list_of_choices, default=...) is a multi-tier CASE WHEN. Conditions are evaluated in order and the first true one wins, just like in SQL. It's the equivalent of Excel's IFS.

  3. .map(dictionary) replaces each value in a Series with the corresponding value from the dictionary. It's a simple VLOOKUP when the lookup table fits in a dictionary. Values not in the dictionary become NaN, so make sure all keys are covered.

  4. .apply(function) applies a function to each value in a Series. With df.apply(function, axis=1), the function receives the entire row, allowing you to access multiple columns, such as row["qty"]. axis=1 means "one row at a time."

  5. Preference order, from fastest to slowest: direct column operations, then np.where and np.select, then map, and last apply. Use apply only when the logic doesn't fit the others. With millions of rows, the difference is huge.

# EXAMPLE

In [1]:
import pandas as pd
import numpy as np

students = pd.DataFrame({
    "name": ["Lia", "Davi", "Rita", "Caio"],
    "major": ["ADS", "ADS", "Law", "Medicine"],
    "grade": [8.5, 4.0, 6.5, 9.2],
})

# 1) np.where: two outcomes
students["passed"] = np.where(students["grade"] >= 6, "yes", "no")

# 2) np.select: multiple tiers (order matters, just like CASE WHEN)
conditions = [students["grade"] >= 9, students["grade"] >= 6]
values = ["excellent", "good"]
students["rating"] = np.select(conditions, values, default="insufficient")

# 3) map: lookup via dictionary
fields = {"ADS": "STEM", "Law": "humanities", "Medicine": "health"}
students["field"] = students["major"].map(fields)

# 4) apply on a column: function receiving a single value
def initial(name):
    return name[0]

students["initial"] = students["name"].apply(initial)

# 5) apply row-wise (axis=1): function receiving the entire row
def summary(row):
    return f"{row['name']} ({row['major']}): {row['grade']}"

students["summary"] = students.apply(summary, axis=1)

In [2]:
print(students)

   name     major  grade passed        rating       field initial  \
0   Lia       ADS    8.5    yes          good        STEM       L   
1  Davi       ADS    4.0     no  insufficient        STEM       D   
2  Rita       Law    6.5    yes          good  humanities       R   
3  Caio  Medicine    9.2    yes     excellent      health       C   

                summary  
0        Lia (ADS): 8.5  
1       Davi (ADS): 4.0  
2       Rita (Law): 6.5  
3  Caio (Medicine): 9.2  


# Exercises


In [8]:
import pandas as pd
import numpy as np

data = {
    "order": [101, 102, 103, 104, 105, 106, 107, 108],
    "customer": ["Ana", "Bruno", "Carla", "Ana", "Diego", "Bruno", "Elisa", "Diego"],
    "city": ["Rio", "Niterói", "Rio", "Rio", "São Gonçalo", "Niterói", "Rio", "São Gonçalo"],
    "product": ["t-shirt", "sneakers", "cap", "pants", "t-shirt", "cap", "sneakers", "pants"],
    "qty": [2, 1, 3, 1, 5, 2, 1, 2],
    "unit_price": [49.9, 249.9, 35.0, 119.9, 49.9, 35.0, 249.9, 119.9],
}

df = pd.DataFrame(data)
df["revenue"] = df["qty"] * df["unit_price"]

In [13]:
# 1. free_shipping: "yes" if revenue is greater than or equal to 100, and "no"
# otherwise. Use np.where.

df["free_shipping"] = np.where(df["revenue"] >= 100, "yes", "no")
print(df)

   order customer         city   product  qty  unit_price  revenue  \
0    101      Ana          Rio   t-shirt    2        49.9     99.8   
1    102    Bruno      Niterói  sneakers    1       249.9    249.9   
2    103    Carla          Rio       cap    3        35.0    105.0   
3    104      Ana          Rio     pants    1       119.9    119.9   
4    105    Diego  São Gonçalo   t-shirt    5        49.9    249.5   
5    106    Bruno      Niterói       cap    2        35.0     70.0   
6    107    Elisa          Rio  sneakers    1       249.9    249.9   
7    108    Diego  São Gonçalo     pants    2       119.9    239.8   

  free_shipping  
0            no  
1           yes  
2           yes  
3           yes  
4           yes  
5            no  
6           yes  
7           yes  


In [14]:
# 2. tier: "large" if revenue is greater than or equal to 200, "medium" if it's
# greater than or equal to 80, and "small" for the rest. Use np.select.

conditions = [df["revenue"] >= 200, df["revenue"] >= 80]
values = ["large", "medium"]
df["tier"] = np.select(conditions, values, default="small")

print(df)


   order customer         city   product  qty  unit_price  revenue  \
0    101      Ana          Rio   t-shirt    2        49.9     99.8   
1    102    Bruno      Niterói  sneakers    1       249.9    249.9   
2    103    Carla          Rio       cap    3        35.0    105.0   
3    104      Ana          Rio     pants    1       119.9    119.9   
4    105    Diego  São Gonçalo   t-shirt    5        49.9    249.5   
5    106    Bruno      Niterói       cap    2        35.0     70.0   
6    107    Elisa          Rio  sneakers    1       249.9    249.9   
7    108    Diego  São Gonçalo     pants    2       119.9    239.8   

  free_shipping    tier  
0            no  medium  
1           yes   large  
2           yes  medium  
3           yes  medium  
4           yes   large  
5            no   small  
6           yes   large  
7           yes   large  


In [16]:
# 3. category: t-shirt and pants become "clothing", cap becomes "accessory", and sneakers becomes
# "footwear". Use map with a dictionary.

outfit = {"t-shirt": "clothing", "pants": "clothing", "cap": "accessory", "sneakers": "footwear"}
df["category"] = df["product"].map(outfit)

print(df)

   order customer         city   product  qty  unit_price  revenue  \
0    101      Ana          Rio   t-shirt    2        49.9     99.8   
1    102    Bruno      Niterói  sneakers    1       249.9    249.9   
2    103    Carla          Rio       cap    3        35.0    105.0   
3    104      Ana          Rio     pants    1       119.9    119.9   
4    105    Diego  São Gonçalo   t-shirt    5        49.9    249.5   
5    106    Bruno      Niterói       cap    2        35.0     70.0   
6    107    Elisa          Rio  sneakers    1       249.9    249.9   
7    108    Diego  São Gonçalo     pants    2       119.9    239.8   

  free_shipping    tier   category  
0            no  medium   clothing  
1           yes   large   footwear  
2           yes  medium  accessory  
3           yes  medium   clothing  
4           yes   large   clothing  
5            no   small  accessory  
6           yes   large   footwear  
7           yes   large   clothing  


In [17]:
# 4. sale_type: create a function classify_qty(qty) that returns "wholesale" if qty >= 5 and
# "retail" otherwise, and apply it to the qty column with apply.

def classify_qty(qty):
  return "atacado" if qty >= 5 else "varejo"

df["sale_type"] = df["qty"].apply(classify_qty)

print(df)

   order customer         city   product  qty  unit_price  revenue  \
0    101      Ana          Rio   t-shirt    2        49.9     99.8   
1    102    Bruno      Niterói  sneakers    1       249.9    249.9   
2    103    Carla          Rio       cap    3        35.0    105.0   
3    104      Ana          Rio     pants    1       119.9    119.9   
4    105    Diego  São Gonçalo   t-shirt    5        49.9    249.5   
5    106    Bruno      Niterói       cap    2        35.0     70.0   
6    107    Elisa          Rio  sneakers    1       249.9    249.9   
7    108    Diego  São Gonçalo     pants    2       119.9    239.8   

  free_shipping    tier   category sale_type  
0            no  medium   clothing    varejo  
1           yes   large   footwear    varejo  
2           yes  medium  accessory    varejo  
3           yes  medium   clothing    varejo  
4           yes   large   clothing   atacado  
5            no   small  accessory    varejo  
6           yes   large   footwear    va

In [20]:
# 5. summary: one sentence per order in the format "Ana bought 2x t-shirt in Rio",
# using apply with axis=1.

def summary(row):
  return f"{row['customer']} comprou {row['qty']}x {row['product']} em {row['city']}"

df["summary"] = df.apply(summary, axis=1)

print(df)

   order customer         city   product  qty  unit_price  revenue  \
0    101      Ana          Rio   t-shirt    2        49.9     99.8   
1    102    Bruno      Niterói  sneakers    1       249.9    249.9   
2    103    Carla          Rio       cap    3        35.0    105.0   
3    104      Ana          Rio     pants    1       119.9    119.9   
4    105    Diego  São Gonçalo   t-shirt    5        49.9    249.5   
5    106    Bruno      Niterói       cap    2        35.0     70.0   
6    107    Elisa          Rio  sneakers    1       249.9    249.9   
7    108    Diego  São Gonçalo     pants    2       119.9    239.8   

  free_shipping    tier   category sale_type                          summary  
0            no  medium   clothing    varejo            Ana comprou 2x em Rio  
1           yes   large   footwear    varejo      Bruno comprou 1x em Niterói  
2           yes  medium  accessory    varejo          Carla comprou 3x em Rio  
3           yes  medium   clothing    varejo     